# Olist E-Commerce Data Analysis

Goal:
Analyze business performance, customer behavior, and delivery efficiency using SQL + DuckDB.

In [1]:
import os
import duckdb

os.getcwd()

'C:\\Users\\Hp'

In [2]:
con = duckdb.connect(r"D:\DataQuartz\Kaggle Dataset Practice\Ecommerce Project\ecommerce.duckdb")

## Step 1.1 — Dataset Overview

### Objective
Understand the overall dataset structure, including available tables and their purpose.

In [4]:
con.execute("SHOW TABLES").df()

,name
0,category_translation
1,customers
2,geolocation
3,order_items
4,orders
5,payments
6,products
7,reviews
8,sellers


## Step 1.2 — Orders Table Structure

### Objective
Examine the schema of the orders table to understand available columns and data types.

In [7]:
con.execute("DESCRIBE orders").df()

,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,customer_id,VARCHAR,YES,None,None,None
2,order_status,VARCHAR,YES,None,None,None
3,order_purchase_timestamp,TIMESTAMP,YES,None,None,None
4,order_approved_at,TIMESTAMP,YES,None,None,None
5,order_delivered_carrier_date,TIMESTAMP,YES,None,None,None
6,order_delivered_customer_date,TIMESTAMP,YES,None,None,None
7,order_estimated_delivery_date,TIMESTAMP,YES,None,None,None


## Step 1.3 — Orders Data Preview

### Objective
Inspect sample records from the orders table to understand real data patterns.

In [8]:
#Total Orders

orders = con.execute("SELECT * FROM orders LIMIT 5").df();
orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


## Step 1.4 — Orders Uniqueness Check

### Objective
Verify whether order_id uniquely identifies each order.

In [9]:
con.execute("""SELECT
        COUNT(*) AS Total_Orders, COUNT(DISTINCT order_id) as Unique_Orders
        FROM orders""").df()
            

,Total_Orders,Unique_Orders
0,99441,99441


## Step 1.5 — Order Items Table Structure

### Objective
Understand the schema of the order_items table.

In [10]:
con.execute("DESCRIBE order_items").df()

,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,order_item_id,BIGINT,YES,None,None,None
2,product_id,VARCHAR,YES,None,None,None
3,seller_id,VARCHAR,YES,None,None,None
4,shipping_limit_date,TIMESTAMP,YES,None,None,None
5,price,DOUBLE,YES,None,None,None
6,freight_value,DOUBLE,YES,None,None,None


## Step 1.6 — Order Items Volume Check

### Objective
Compare total rows vs unique orders to understand item-level granularity.

In [11]:
con.execute("""SELECT 
            COUNT(*) AS total_order_items, COUNT(DISTINCT order_id) as Unique_Orders
            FROM order_items""").df()

,total_order_items,Unique_Orders
0,112650,98666


## Step 1.7 — Orders & Order Items Relationship

### Objective
Understand how orders relate to order_items using joins.

In [12]:
# Let's JOIN Orders with Order_Items

con.execute("""SELECT *
                FROM orders i
                LEFT JOIN order_items o
                ON i.order_id = o.order_id
                LIMIT 10""").df()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_id_1,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
5,00048cc3ae777c65dbb7d2a0634bc1ea,816cbea969fe5b689b39cfc97a506742,delivered,2017-05-15 21:42:34,2017-05-17 03:55:27,2017-05-17 11:05:55,2017-05-22 13:44:35,2017-06-06,00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23 03:55:27,21.90,12.69
6,00054e8431b9d7675808bcb819fb4a32,32e2e6ab09e778d99bf2e0ecd4898718,delivered,2017-12-10 11:53:48,2017-12-10 12:10:31,2017-12-12 01:07:48,2017-12-18 22:03:38,2018-01-04,00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14 12:10:31,19.90,11.85
7,000576fe39319847cbb9d288c5617fa6,9ed5e522dd9dd85b4af4a077526d8117,delivered,2018-07-04 12:08:27,2018-07-05 16:35:48,2018-07-05 12:15:00,2018-07-09 14:04:07,2018-07-25,000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10 12:30:45,810.00,70.75
8,0005a1a1728c9d785b8e2b08b904576c,16150771dfd4776261284213b89c304e,delivered,2018-03-19 18:40:33,2018-03-20 18:35:21,2018-03-28 00:37:42,2018-03-29 18:17:31,2018-03-29,0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26 18:31:29,145.95,11.65
9,0005f50442cb953dcd1d21e1fb923495,351d3cb2cee3c7fd0af6616c82df21d3,delivered,2018-07-02 13:59:39,2018-07-02 14:10:56,2018-07-03 14:25:00,2018-07-04 17:28:31,2018-07-23,0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06 14:10:56,53.99,11.40


## Step 1.8 — Customers Table Preview

### Objective
Explore customer-level data including location information.

In [13]:
#Total Customers 

customers = con.execute("SELECT * FROM customers LIMIT 5").df();
customers

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


## Step 1.9 — Payments Table Preview

### Objective
Analyze payment structure and transaction behavior.

In [14]:
#Total payments 

payments = con.execute("SELECT * FROM payments LIMIT 5").df();
payments

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


## Step 1.10 — Payment Behavior Analysis

### Objective
Identify orders with multiple payment entries.

In [57]:
con.execute("""SELECT order_id, COUNT(*) as Total_Payments
FROM payments
GROUP BY order_id
ORDER BY COUNT(*) DESC
LIMIT 5""").df()

,order_id,Total_Payments
0,fa65dad1b0e818e3ccc5cb0e39231352,29
1,ccf804e764ed5650cd8759557269dc13,26
2,285c2e15bebd4ac83635ccc563dc71f4,22
3,895ab968e7bb0d5659d16cd74cd1650c,21
4,fedcd9f7ccdc8cba3a18defedd1a5547,19


## Step 1.11 — Reviews Table Preview

### Objective
Understand customer feedback data structure.

In [16]:
#Total reviews 

reviews = con.execute("SELECT * FROM reviews LIMIT 5").df();
reviews

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,None,None,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,None,None,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,None,None,2018-02-17,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,None,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,None,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53


## Step 1.12 — Reviews Coverage Check

### Objective
Check how many orders have associated reviews.

In [56]:
con.execute("""SELECT COUNT(*) as Toal_Reviews, COUNT(DISTINCT order_id) as unique_order_id
FROM reviews""").df()

,Toal_Reviews,unique_order_id
0,99224,98673


## Step 1.13 — Sellers Table Preview

### Objective
Explore seller-level geographic data.

In [18]:
#Total sellers 

sellers = con.execute("SELECT * FROM sellers LIMIT 5").df();
sellers

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


## Step 1.14 — Geolocation Data

### Objective
Understand location mapping data for customers and sellers.

In [19]:
#Total geolocation 

geolocation = con.execute("SELECT * FROM geolocation LIMIT 5").df();
geolocation

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01037,-23.545621,-46.639292,sao paulo,SP
1,01046,-23.546081,-46.644820,sao paulo,SP
2,01046,-23.546129,-46.642951,sao paulo,SP
3,01041,-23.544392,-46.639499,sao paulo,SP
4,01035,-23.541578,-46.641607,sao paulo,SP


## Step 1.15 — Category Translation

### Objective
Map product categories from Portuguese to English.

In [20]:
#Total category_translation 

category_translation = con.execute("SELECT * FROM category_translation LIMIT 5").df();
category_translation

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


## Step 1.16 — Products Table

### Objective
Understand product attributes and category information.

In [22]:
#Total products 

customers = con.execute("SELECT * FROM products LIMIT 5").df();
customers

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15
3,cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13


## Step 1.17 — Revenue by Product Category

### Objective
Calculate revenue contribution of each product category.

In [23]:
#JOIN Order_Items + Products to find Revenue for each product

con.execute("""SELECT COALESCE(p.product_category_name, 'unknown') as Product_Category, SUM(price + freight_value) as Revenue
            FROM order_items o
            LEFT JOIN products p
            ON o.product_id = p.product_id
            GROUP BY product_category_name
            ORDER BY Revenue DESC
            LIMIT 5""").df()
            

,Product_Category,Revenue
0,beleza_saude,1441248.07
1,relogios_presentes,1305541.61
2,cama_mesa_banho,1241681.72
3,esporte_lazer,1156656.48
4,informatica_acessorios,1059272.40


## Step 1 — Conclusion

The dataset consists of multiple interconnected tables representing orders, customers, products, payments, and logistics.

Key observations:
- Each order is uniquely identified by order_id
- One order can have multiple items (1-to-many relationship)
- Payments can have multiple entries per order
- Reviews are available for most orders but not all
- Product categories require translation for better readability

This structure enables end-to-end business analysis from order placement to delivery and customer feedback.

# STEP 2 DATA CLEANING:


## Step 2.1 — Missing Values Check (Order Items)

### Objective
Verify whether critical columns contain NULL values for order items table.

In [24]:
order_items = con.execute("""SELECT COUNT(*) as total_rows, COUNT( order_id) as total_order_items, COUNT( price) as total_price,
                            COUNT( freight_value) as Total_Freight
                            FROM order_items""").df()
order_items

,total_rows,total_order_items,total_price,Total_Freight
0,112650,112650,112650,112650


#### Conclusion:
All key columns contain no missing values, confirming complete and reliable transaction-level data.

## Step 2.2 — Missing Values Check (Orders)

### Objective
Check delivery-related fields to understand missing value behavior.

In [59]:
#Total Orders

orders = con.execute("SELECT * FROM orders LIMIT 5").df();
orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [60]:
orders = con.execute("""SELECT COUNT(*) as Total_Rows, COUNT(order_id) as Total_Orders, COUNT(order_delivered_customer_date) as Delivery_Date,
                        COUNT(order_estimated_delivery_date) AS Estimated_Date, COUNT(order_status) as Total_Order_Status
                        FROM orders""").df()

orders

,Total_Rows,Total_Orders,Delivery_Date,Estimated_Date,Total_Order_Status
0,99441,99441,96476,99441,99441


#### Conclusion:
Missing delivery timestamps are expected for non-delivered orders and do not indicate data quality issues.

### Objective
Analyze order status distribution to support interpretation of missing delivery fields.

In [27]:
con.execute("""SELECT order_status, COUNT(*) as Total_Order_Status
                FROM orders
                GROUP BY order_status
                ORDER BY Total_Order_Status DESC""").df()

,order_status,Total_Order_Status
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


#### Conclusion
Most orders are delivered, while a smaller portion remains in non-delivered states, explaining missing delivery timestamps.

### Step 2.3 — Duplicate Checks

### Objective
Ensure that each order_id uniquely identifies records in the orders table.

In [28]:
con.execute("""SELECT order_id, COUNT(*)
                FROM orders
                GROUP BY order_id
                HAVING COUNT(*) > 1""").df()

,order_id,count_star()


#### Conclusion
No duplicate records found, confirming primary key integrity.

### Objective
Verify uniqueness of order_id and order_item_id combinations in the order_items table.

In [29]:
con.execute("""SELECT order_id, order_item_id, COUNT(*)
                FROM order_items
                GROUP BY order_id, order_item_id
                HAVING COUNT(*) > 1""").df()

,order_id,order_item_id,count_star()


#### Conclusion
No duplicate combinations found, confirming item-level uniqueness.

### Step 2.4 — Join Integrity

### Objective
Validate that all order items have corresponding records in the orders table.

In [30]:
con.execute("""SELECT oi.order_id, oi.order_item_id, COUNT(*) as total_orders
                FROM order_items oi
                LEFT JOIN orders o
                ON oi.order_id = o.order_id
                WHERE o.order_id IS NULL
                GROUP BY oi.order_id, oi.order_item_id""").df()
                

,order_id,order_item_id,total_orders


#### Conclusion
No orphan records/orders found, confirming strong referential integrity.

### Step 2.5 — Value Sanity Checks

### Objective
Verify that price and freight values fall within valid and realistic business ranges.

In [31]:
con.execute("""SELECT MIN(price), MIN(freight_value), MAX(price), MAX(freight_value), AVG(price), AVG(freight_value)
                FROM order_items""").df()

,min(price),min(freight_value),max(price),max(freight_value),avg(price),avg(freight_value)
0,0.85,0.0,6735.0,409.68,120.653739,19.99032


### Conclusion:

Price and freight values fall within reasonable business ranges. No negative or invalid values detected, indicating consistent revenue-related data.
We checked the average to see the outlier, Outliers increase the average significantly, while the median remains more representative of the typical value.

### Step 2.6 — Time Sanity Checks

### Objective
Ensure delivery durations follow logical patterns and realistic timelines.

In [32]:
#Total Orders

orders = con.execute("SELECT * FROM orders LIMIT 5").df();
orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


#### Conclusion
Average delivery time is approximately 12.5 days.

### Objective
Compare actual delivery time with estimated delivery timelines.

In [6]:
con.execute("""SELECT AVG(DATEDIFF('day', order_purchase_timestamp, order_delivered_customer_date)) as Actual_Delivery_Time
            FROM orders
            WHERE order_delivered_customer_date IS NOT NULL
            """).df()

,Actual_Delivery_Time
0,12.497336


#### Conclusion
Orders are delivered earlier than estimated on average, indicating conservative delivery estimates.

Average delivery time ≈ 12.5 days

### Objective
Classify deliveries into early, on-time, and late categories.

In [7]:
con.execute("""SELECT AVG(DATEDIFF('day', order_estimated_delivery_date, order_delivered_customer_date)) AS Estimated_Delivery_Time
                FROM orders
                WHERE order_delivered_customer_date IS NOT NULL""").df()

,Estimated_Delivery_Time
0,-11.876881


#### Conclusion: 
On average, orders are delivered ~11.8 days earlier than the estimated delivery date, indicating that delivery estimates are highly conservative.

### Objective
Classify deliveries into early, on-time, and late categories.

In [11]:
con.execute("""SELECT COUNT(*) * 100/SUM(COUNT(*)) OVER() as Total_Orders_Percent,
                        CASE 
                        WHEN DATEDIFF('day', order_estimated_delivery_date, order_delivered_customer_date) < 0
                        THEN 'early'
                        WHEN DATEDIFF('day', order_estimated_delivery_date, order_delivered_customer_date) = 0
                        THEN 'On-Time'
                        WHEN DATEDIFF('day', order_estimated_delivery_date, order_delivered_customer_date) > 0 
                        THEN 'late' END as Average_Delay
                        FROM orders
                        WHERE order_delivered_customer_date IS NOT NULL
                        GROUP BY Average_Delay
                        ORDER BY Total_Orders_Percent DESC""").df()

,Total_Orders_Percent,Average_Delay
0,91.887101,early
1,6.773705,late
2,1.339193,On-Time


#### Conclusion: 
Over 90% of orders are delivered earlier than the estimated date, while only 7% are delayed


### Step 2.7 — Business Rule Validation

### Objective
Validate whether delivery timelines follow correct chronological order.

#### Rule 1:
Can an order be delivered before it was purchased?

In [36]:
con.execute("""SELECT COUNT(*)
FROM orders
WHERE order_delivered_customer_date < order_purchase_timestamp
""").df()

,count_star()
0,0


#### Conclusion
No violations found, confirming correct sequence.

In [37]:
#Total Orders (again for reference)

orders = con.execute("SELECT * FROM orders LIMIT 5").df();
orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


#### Rule 2:
Delivered orders must have delivery date

In [38]:
con.execute("""SELECT COUNT(*)
                FROM orders
                WHERE order_status = 'delivered' AND order_delivered_customer_date IS NULL""").df()

,count_star()
0,8


#### Conclusion
A very small number of inconsistencies were found, likely due to data entry issues.

#### Rule 3:
Non-delivered orders should NOT have delivery date

In [39]:
con.execute("""SELECT Count(*)
                FROM orders
                WHERE order_status != 'delivered' AND order_delivered_customer_date IS NOT NULL""").df()

,count_star()
0,6


#### Conclusion
Minor inconsistencies observed, indicating slight issues in status tracking.

### Step 2.8 — Distribution & Outlier Analysis

### Objective
Analyze price distribution to identify skewness and extreme values.

In [40]:
#Total Orders Items (again for reference)

order_items = con.execute("SELECT * FROM order_items LIMIT 5").df();
order_items

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [41]:
con.execute("""SELECT 
                    percentile_cont(0.5) WITHIN GROUP (ORDER BY price) as P55,
                    percentile_cont(0.9) WITHIN GROUP (ORDER BY price) as P90,
                    percentile_cont(0.95) WITHIN GROUP (ORDER BY price) as P95,
                    percentile_cont(0.99) WITHIN GROUP (ORDER BY price) as P99
                    
                FROM order_items""").df()

,P55,P90,P95,P99
0,74.99,229.8,349.9,890.0


#### Conclusion
The price distribution is heavily right-skewed, with the majority of products concentrated under 350
and a small number of extreme outliers significantly increasing the maximum value.

### Objective
Identify high-value outliers and evaluate their frequency.

In [42]:
con.execute("""SELECT product_id, COUNT(*) as Outliers_Count, COUNT(*) * 100/ SUM(COUNT(*)) OVER() as percentage
                FROM order_items
                WHERE price > 900
                GROUP BY product_id
                ORDER BY Outliers_Count DESC""").df()

,product_id,Outliers_Count,percentage
0,25c38557cf793876c5abdd5931f922db,38,3.696498
1,d6160fb7873f184099d9bc95e30376af,35,3.404669
2,588531f8ec37e7d5ff5b7b22ea0488f8,20,1.945525
3,a5215a7a9f46c4185b12f38e9ddf2abc,17,1.653696
4,bc4cd4da98dd128c39bf0b8c2674032f,17,1.653696
...,...,...,...
545,d48c8d7a322bda722704d0ba7506e9a1,1,0.097276
546,4abee1df902ca6e48fbe864fce3859bc,1,0.097276
547,da066b813a9bc3f4294eac9aa647c834,1,0.097276
548,ca8abbdcac2d082a56ff54df35aec76a,1,0.097276


#### Conclusion
Some expensive products sell repeatedly, while many expensive items appear only once

### Step 2.9 — Cross-Table Consistency Checks

### Objective
Calculate total order value from item-level data.

In [43]:
con.execute("""SELECT order_id, ROUND(SUM(price + freight_value),2) as Total_Price
                FROM order_items
                GROUP BY order_id""").df()

,order_id,Total_Price
0,000229ec398224ef6ca0657da4fc703e,216.87
1,0006ec9db01a64e59a68b2c340bf65a7,97.32
2,00664284de7a3470d22931ed78615ee4,55.75
3,008c3a655c66f4d92370d0d422732f69,99.44
4,00d4439957b7fd3f8da4a33a965817fb,33.97
...,...,...
98661,ff3d961f941d0784936dbe5da739575e,144.06
98662,ff54387ebb9660cb3ffc874139e4bd56,155.03
98663,ffa93f1bd7c0039af32324fa5d3cf1e6,37.61
98664,ffda7f88e6a571a9e73a0c9c778e606d,51.59


### Objective
Aggregate total payments per order.

In [45]:
con.execute("""SELECT order_id, SUM(payment_value) as Total_Payment
                FROM payments
                GROUP BY order_id""").df()

,order_id,Total_Payment
0,a9810da82917af2d9aefd1278f1dcfa0,24.39
1,8ac09207f415d55acff302df7d6a895c,244.15
2,5d9c5817e278892b7498d90bfa28ade8,290.16
3,11978d520d85578c9f024b99ac1a87ef,50.09
4,c776863a93dc0740c6e7d78104b21413,264.08
...,...,...
99435,e137f3b8d8f48a149df7322275e474c5,43.93
99436,9e3239dcd0d89b2b4e9315e93f265c84,82.12
99437,16b2fea16c665281ed07933ee9b9f095,91.19
99438,8da1b0cac91830c54472e2f91579aeff,186.27


### Objective
Compare order values with payment values to ensure consistency.

In [46]:
con.execute("""WITH order_items_total AS (

                SELECT order_id, ROUND(SUM(price + freight_value),2) as Total_Price
                FROM order_items
                GROUP BY order_id),
                
                payments_total AS (

                SELECT order_id, SUM(payment_value) as Total_Payment
                FROM payments
                GROUP BY order_id )

                SELECT o.order_id,
                o.Total_Price,
                p.Total_Payment,
                o.Total_Price - p.Total_Payment as diff
                FROM order_items_total o
                LEFT JOIN payments_total p
                ON o.order_id = p.order_id""").df()


,order_id,Total_Price,Total_Payment,diff
0,a9810da82917af2d9aefd1278f1dcfa0,24.39,24.39,0.0
1,8ac09207f415d55acff302df7d6a895c,244.15,244.15,0.0
2,5d9c5817e278892b7498d90bfa28ade8,290.16,290.16,0.0
3,11978d520d85578c9f024b99ac1a87ef,50.09,50.09,0.0
4,c776863a93dc0740c6e7d78104b21413,264.08,264.08,0.0
...,...,...,...,...
98661,f255802c9fa377d64b1699bcf855a608,54.00,54.00,0.0
98662,7e3db8d87e79e50d7a08312194e1cf01,164.46,164.46,0.0
98663,807f4ef550f104708e2190eca7e4c37a,205.71,205.71,0.0
98664,d5639d8a973cec351be75a3186835068,160.89,160.89,0.0


#### Conclusion:
Order-level revenue is consistent across order_items and payments, with no significant discrepancies observed

## Step 2 — Conclusion

The dataset demonstrates strong data quality across all validation checks.

- No missing values in critical transactional fields  
- No duplicate records in primary or composite keys  
- No orphan records, confirming referential integrity  
- Revenue values fall within valid business ranges  
- Delivery timelines are consistent and logically ordered  
- Minor inconsistencies observed are negligible  
- Price distribution shows expected right-skew with manageable outliers  
- Strong consistency between order values and payments  

Overall, the dataset is clean, reliable, and ready for analysis.

### Step 3.1 — Revenue by Category

### Objective
Identify top revenue-generating product categories.

### Approach
Join order_items with products, calculate revenue as (price + freight_value), and aggregate by category.

Which product categories generate the most revenue?

In [47]:
#Total order_items (again for reference)

order_items = con.execute("SELECT * FROM order_items LIMIT 5").df();
order_items

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [48]:
#Total products (again for reference)

customers = con.execute("SELECT * FROM products LIMIT 5").df();
customers

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15
3,cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13


### Objective:

Identify top revenue-generating categories

### Approach:
Join tables, revenue = price + freight, group by category

Which product categories generate the most revenue?

In [49]:
Products_Revenue = con.execute("""SELECT p.product_category_name, ROUND(SUM(o.price + o.freight_value),2) AS Revenue
                FROM order_items o
                LEFT JOIN products p
                ON o.product_id = p.product_id
                GROUP BY p.product_category_name
                ORDER BY Revenue DESC""").df()

Products_Revenue

,product_category_name,Revenue
0,beleza_saude,1441248.07
1,relogios_presentes,1305541.61
2,cama_mesa_banho,1241681.72
3,esporte_lazer,1156656.48
4,informatica_acessorios,1059272.40
...,...,...
69,flores,1598.91
70,casa_conforto_2,1170.58
71,cds_dvds_musicais,954.99
72,fashion_roupa_infanto_juvenil,665.36


#### Conclusion
Revenue is distributed across multiple categories with no single dominant category. The top categories contribute similar revenue levels, indicating a balanced product distribution.

### Step 3.2 — Revenue Distribution (%) 

### Objective
Measure each category’s contribution to total revenue.

### Approach
Use a window function to calculate the percentage contribution of each category.

Do top categories dominate total revenue?

In [50]:
Revenue_Percentage = con.execute("""SELECT product_category_name, Revenue,
                Revenue * 100 / SUM(Revenue) OVER() as Percentage
                FROM Products_Revenue""").df()

Revenue_Percentage

,product_category_name,Revenue,Percentage
0,beleza_saude,1441248.07,9.096748
1,relogios_presentes,1305541.61,8.240207
2,cama_mesa_banho,1241681.72,7.837142
3,esporte_lazer,1156656.48,7.300487
4,informatica_acessorios,1059272.40,6.685826
...,...,...,...
69,flores,1598.91,0.010092
70,casa_conforto_2,1170.58,0.007388
71,cds_dvds_musicais,954.99,0.006028
72,fashion_roupa_infanto_juvenil,665.36,0.004200


#### Conclusion
Top categories contribute approximately 6–9% each, indicating a well-balanced revenue distribution. The top 5 categories contribute around 40% of total revenue, showing moderate concentration but no heavy dominance.

### Step 3.3 — Cumulative Revenue (Pareto Analysis 80/20)

### Objective
Analyze cumulative revenue contribution across categories.

### Approach
Use a running total (window function) to calculate cumulative percentage ordered by revenue.

How many categories contribute to ~80% of total revenue?

In [51]:
Cumulative_Percentage = con.execute("""SELECT product_category_name, Revenue, percentage,
                                        SUM(percentage) OVER(ORDER BY Revenue DESC) AS Cumulative_Percentage
                                        FROM Revenue_Percentage""").df()

Cumulative_Percentage

,product_category_name,Revenue,Percentage,Cumulative_Percentage
0,beleza_saude,1441248.07,9.096748,9.096748
1,relogios_presentes,1305541.61,8.240207,17.336955
2,cama_mesa_banho,1241681.72,7.837142,25.174097
3,esporte_lazer,1156656.48,7.300487,32.474583
4,informatica_acessorios,1059272.40,6.685826,39.160409
...,...,...,...,...
69,flores,1598.91,0.010092,99.980336
70,casa_conforto_2,1170.58,0.007388,99.987725
71,cds_dvds_musicais,954.99,0.006028,99.993752
72,fashion_roupa_infanto_juvenil,665.36,0.004200,99.997952


#### Conclusion
Revenue accumulates gradually across categories with no sharp dominance, indicating a distributed contribution pattern rather than a classic Pareto-heavy skew.

### Step 3.4 — 80% Revenue Threshold

### Objective
Identify categories contributing to 80% of total revenue.

### Approach
Filter cumulative percentage to values less than or equal to 80%.

Which categories form the core revenue drivers?

In [52]:
con.execute("""SELECT product_category_name, Revenue, percentage, Cumulative_Percentage
                FROM Cumulative_Percentage
                WHERE Cumulative_Percentage <= 80""").df()

,product_category_name,Revenue,Percentage,Cumulative_Percentage
0,beleza_saude,1441248.07,9.096748,9.096748
1,relogios_presentes,1305541.61,8.240207,17.336955
2,cama_mesa_banho,1241681.72,7.837142,25.174097
3,esporte_lazer,1156656.48,7.300487,32.474583
4,informatica_acessorios,1059272.40,6.685826,39.160409
5,moveis_decoracao,902511.79,5.696398,44.856807
6,utilidades_domesticas,778397.77,4.913025,49.769832
7,cool_stuff,719329.95,4.540206,54.310038
8,automotivo,685384.32,4.325951,58.635989
9,ferramentas_jardim,584219.21,3.687425,62.323414


#### Conclusion
Approximately 17 categories contribute to 80% of total revenue, indicating moderate concentration with a long-tail distribution.

### Step 3.5 — Final Insight

#### Conclusion

Revenue follows a moderate long-tail distribution rather than being highly concentrated.

Approximately the top 17 categories contribute around 80% of total revenue, indicating that multiple categories play a significant role instead of a few dominant ones.

## Step 4 — Customer & Order Behavior Analysis

### Objective
Categorize orders based on total order value.

### Approach
Calculate total order value per order (price + freight), then bucket into Low, Medium, and High value groups.

How are orders distributed across different value segments?

In [53]:
#Total order_items (again for reference)

order_items = con.execute("SELECT * FROM order_items LIMIT 5").df();
order_items

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [73]:
Total_Order_Value = con.execute(""" 
SELECT CASE
                    WHEN Total_Value > 300 THEN 'High_Value'
                    WHEN Total_Value BETWEEN 100 AND 300 THEN 'Medium_Value'
                    ELSE 'Low_Value' END AS Bucket,
                    Count(*)  as order_count
FROM (
        SELECT order_id, SUM(price + freight_value) AS Total_Value
        FROM order_items
        GROUP BY order_id)
        
            GROUP BY Bucket
            ORDER BY order_count DESC""").df()

Total_Order_Value

,Bucket,order_count
0,Low_Value,46902
1,Medium_Value,41552
2,High_Value,10212


#### Conclusion
Most orders fall into the low and medium value categories, indicating that customers generally make smaller purchases. High-value orders are significantly fewer, suggesting an opportunity to increase average order value through upselling or bundling strategies.

In [84]:
#PRACTICE QUERY

con.execute("""SELECT CASE WHEN Freight_Value = 0 THEN 'Free Shipping'
                            ELSE 'Paid Shipping' END as Shipping_Status,
                            COUNT(*) AS Bucket
FROM (
        SELECT order_id, SUM(freight_value) AS Freight_Value
                FROM order_items
                GROUP BY order_id)
        GROUP BY Shipping_Status
        Order by Bucket DESC
                
                """).df()

,Shipping_Status,Bucket
0,Paid Shipping,98328
1,Free Shipping,338


### Step 4.2 — Repeat vs One-Time Customers
### Objective
Identify whether customers are one-time or repeat buyers.

### Approach
Count number of orders per customer and classify them into one-time or repeat customers.

What proportion of customers return after their first purchase?

In [85]:
#Total Customers (again for reference)

customers = con.execute("SELECT * FROM customers LIMIT 5").df();
customers

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [86]:
#Total Orders (again for reference)

orders = con.execute("SELECT * FROM orders LIMIT 5").df();
orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [13]:
customer_unique_table = con.execute("""SELECT o.order_id, o.customer_id, c.customer_unique_id
                FROM orders o
                LEFT JOIN customers c
                ON o.customer_id = c.customer_id""").df()

customer_unique_table

,order_id,customer_id,customer_unique_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8
3,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6
4,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,80bb27c7c16e8f973207a5086ab329e2
...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,6359f309b166b0196dbf7ad2ac62bb5a
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,da62f9e57a76d978d02ab5362c509660
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,737520a9aad80b3fbbdad19b66b37b30
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,5097a5312c8b157bb7be58ae360ef43c


In [15]:
con.execute("""SELECT CASE WHEN
                            Order_Count = 1 THEN 'One_time_Customer'
                            ELSE 'Repeat_Customer' END as Customer_Type,
                            COUNT(*) AS Customer_Count
FROM (

SELECT customer_unique_id, COUNT(order_id) AS Order_Count
                FROM customer_unique_table
                GROUP BY customer_unique_id)
                GROUP BY Customer_Type""").df()
                

,Customer_Type,Customer_Count
0,One_time_Customer,93099
1,Repeat_Customer,2997


#### Conclusion
Approximately 90% of customers are one-time buyers, while less than 10% are repeat customers, indicating very low customer retention. This suggests the business relies heavily on new customer acquisition and has significant opportunity to improve retention through loyalty programs and engagement strategies.

### Step 4.3 — Customer Lifetime Value (CLV)

### Objective
Identify high-value customers based on total spending.

### Approach
Aggregate total spending per customer and segment customers into low, medium, and high spend groups.

How is customer spending distributed across different value segments?


In [111]:
#Total order_items (again for reference)

order_items = con.execute("SELECT * FROM order_items LIMIT 5").df();
order_items

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [126]:
customer_unique_table = con.execute("""SELECT o.order_id, o.customer_id, c.customer_unique_id
                FROM orders o
                LEFT JOIN customers c
                ON o.customer_id = c.customer_id""").df()

customer_unique_table

,order_id,customer_id,customer_unique_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8
3,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6
4,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,80bb27c7c16e8f973207a5086ab329e2
...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,6359f309b166b0196dbf7ad2ac62bb5a
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,da62f9e57a76d978d02ab5362c509660
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,737520a9aad80b3fbbdad19b66b37b30
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,5097a5312c8b157bb7be58ae360ef43c


In [18]:
con.execute("""SELECT  
                    CASE
                        WHEN Total_Value > 300 THEN 'High_Spend'
                        WHEN Total_Value BETWEEN 100 AND 300 THEN 'Medium_Spend'
                        ELSE 'Low_Spend' END AS total_spend,
                        COUNT(customer_unique_id) AS customer_count
                        
FROM (
    
    SELECT cu.customer_unique_id,
    SUM(o.price + o.freight_value) as Total_Value
    FROM customer_unique_table cu
    LEFT JOIN order_items o
    ON cu.order_id = o.order_id
    GROUP BY cu.customer_unique_id)
    
    GROUP BY total_spend
    ORDER BY total_spend DESC
    """).df()

,total_spend,customer_count
0,Medium_Spend,40634
1,Low_Spend,44800
2,High_Spend,10662


#### Conclusion
Most customers fall into low and medium spend categories, while high-value customers represent a much smaller segment. This indicates revenue is largely driven by lower-value purchases, with an opportunity to grow high-value customer segments through targeted strategies.

### Step 4.4 — Order Frequency

### Objective
Analyze how frequently customers place orders.

### Approach
Count total orders per customer and classify them into low, medium, and high frequency groups.

How often do customers return to make additional purchases?

In [145]:
con.execute("""SELECT  CASE
                        WHEN Total_Orders > 3 THEN 'High_Frequency'
                        WHEN Total_Orders >= 2 AND Total_Orders <= 3 THEN 'Medium_Frequency'
                        WHEN Total_Orders = 1 THEN 'Low_Frequency' END AS Frequency, COUNT(customer_unique_id) as Customer_Count
                        
FROM (

SELECT customer_unique_id, COUNT(order_id) as Total_Orders
                FROM customer_unique_table
                GROUP BY customer_unique_id)
                
                GROUP BY Frequency
                ORDER BY customer_Count DESC
                """).df()

,Frequency,Customer_Count
0,Low_Frequency,93099
1,Medium_Frequency,2948
2,High_Frequency,49


#### Conclusion
The majority of customers place only a single order, while very few customers make multiple purchases. This indicates extremely low customer engagement and retention, highlighting the need for strategies focused on repeat purchases and long-term customer relationships.

## Step 5 — Key Business Metrics (KPIs)

### Step 5.1 — Total Revenue

### Objective
Calculate total revenue generated.

### Approach
Sum total revenue using price + freight_value across all order items.

What is the total revenue generated by the business?

In [146]:
#Total order_items (again for reference)

order_items = con.execute("SELECT * FROM order_items LIMIT 5").df();
order_items

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [154]:
Total_Revenue = con.execute("""SELECT SUM(price + freight_value) as Total_Revenue
                FROM order_items""").df()

Total_Revenue

,Total_Revenue
0,1.584355e+07


#### Conclusion
The company generated approximately $15.8 million in total revenue, including both product prices and shipping charges. This represents the overall top-line performance of the business and serves as a baseline for evaluating growth and customer contribution.

### Step 5.2 — Total Orders

### Objective
Calculate total number of orders placed.

### Approach
Count total number of order_id from the orders table.

What is the total transaction volume of the business?


In [148]:
#Total Orders

orders = con.execute("SELECT * FROM orders LIMIT 5").df();
orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [155]:
Total_Orders = con.execute("""SELECT COUNT(order_id) AS Total_Orders
                FROM Orders""").df()

Total_Orders 

,Total_Orders
0,99441


#### Conclusion
A total of 99,441 orders have been placed, reflecting the overall transaction volume of the business. This provides a key indicator of demand and customer activity.

### Step 5.3 — Total Customers

### Objective
Calculate total number of unique customers.

### Approach
Count distinct customer_unique_id from the customers table.

How large is the active customer base?

In [151]:
con.execute("""SELECT Count(DISTINCT customer_unique_id) as Total_Unique_Customers 
                FROM customers""").df()

,Total_Unique_Customers
0,96096


#### Conclusion
A total of 96,096 unique customers have made purchases, representing the overall customer base. This metric is essential for analyzing customer retention and engagement patterns.

### Step 5.4 — Average Order Value (AOV)

### Objective
Calculate the average revenue generated per order.

### Approach
Divide total revenue by the number of unique orders.

How much revenue does each order generate on average?


In [194]:
con.execute("""SELECT 
    ROUND(SUM(price + freight_value) / COUNT(DISTINCT order_id), 2) AS AOV
FROM order_items;""").df()

,AOV
0,160.58


#### Conclusion
The average order value (AOV) is approximately 160.58, meaning each order generates around 160 in revenue on average. This helps assess purchasing behavior and pricing effectiveness.

### Step 5.5 — Revenue per Customer (ARPU)

### Objective
Calculate the average revenue generated per customer.

### Approach
Divide total revenue by the number of unique customers.

How much revenue does each customer generate on average?

In [195]:
#Total order_items (again for reference)

order_items = con.execute("SELECT * FROM order_items LIMIT 5").df();
order_items

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [203]:
#Total order (again for reference)

orders = con.execute("""SELECT * FROM orders LIMIT 5""").df();
orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [204]:
#Total Customers (again for reference)

Customers = con.execute("""SELECT * FROM Customers LIMIT 5""").df();
Customers

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [20]:
con.execute("""

WITH Order_Level AS (

                    SELECT o.order_id, o.customer_id, SUM(oi.price + oi.freight_value) AS Total_Revenue
                    FROM orders o
                    LEFT JOIN order_items oi
                    ON o.order_id = oi.order_id
                    GROUP BY o.order_id, o.customer_id

                     ),
                
        Customer_Level AS 

                (
                            SELECT ol.order_id, ol.Total_Revenue, c.customer_unique_id
                            FROM Customers c
                            LEFT JOIN Order_Level ol
                            ON c.customer_id = ol.customer_id
                            GROUP BY c.customer_unique_id, ol.order_id, ol.Total_Revenue

                            )

        SELECT SUM(Total_Revenue) / COUNT(DISTINCT customer_unique_id) AS ARPU
        FROM Customer_Level
         ORDER BY ARPU DESC
                """).df()

,ARPU
0,164.872141


#### Conclusion:

The ARPU 164.87 is slightly higher than AOV 160 , indicating that total orders exceed the number of unique customers. This suggests the presence of repeat purchases, although earlier analysis shows that repeat customers form a small proportion of the customer base.

## Step 6 — Time-Based Analysis

### Step 6.1 — Orders Trend Over Time

### Objective
Analyze how order volume changes over time.

### Approach
Group orders by month using order_purchase_timestamp and count total orders.

What are the monthly trends in order volume?

In [5]:
#Total Orders (again for reference)

orders = con.execute("SELECT * FROM orders LIMIT 5").df();
orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [12]:
con.execute("""SELECT strftime('%Y-%m', order_purchase_timestamp) AS Months,
                Count(order_id) AS Total_Orders
                FROM orders
                GROUP BY Months
                ORDER BY Months
                """).df()

,Months,Total_Orders
0,2016-09,4
1,2016-10,324
2,2016-12,1
3,2017-01,800
4,2017-02,1780
5,2017-03,2682
6,2017-04,2404
7,2017-05,3700
8,2017-06,3245
9,2017-07,4026


## Conclusion

Order volume shows a clear upward trend over time, indicating business growth and increasing customer adoption.

There is a steady increase from early periods, followed by stronger growth in later months, suggesting improved market penetration or operational scaling.

Some fluctuations are observed, but the overall trajectory remains positive, reflecting a growing and active customer base.

## Step 6.2 — Revenue Trend Over Time

### Objective
Analyze how revenue changes over time.

### Approach
Aggregate total revenue (price + freight_value) by month.

How does revenue growth compare with order growth over time?

In [13]:
#Total order_items (again for reference)

order_items = con.execute("SELECT * FROM order_items LIMIT 5").df();
order_items

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [21]:
con.execute("""SELECT strftime('%Y-%m', o.order_purchase_timestamp) AS Months,
                SUM(oi.price + oi.freight_value) AS Total_Revenue
                FROM order_items oi
                LEFT JOIN orders o
                ON oi.order_id = o.order_id
                GROUP BY strftime('%Y-%m', o.order_purchase_timestamp)
                ORDER BY strftime('%Y-%m', o.order_purchase_timestamp)""").df()

,Months,Total_Revenue
0,2016-09,354.75
1,2016-10,56808.84
2,2016-12,19.62
3,2017-01,137188.49
4,2017-02,286280.62
5,2017-03,432048.59
6,2017-04,412422.24
7,2017-05,586190.95
8,2017-06,502963.04
9,2017-07,584971.62


#### Conclusion

Revenue shows a clear upward trend over time, closely aligning with the growth in order volume.

Initial months have low revenue due to fewer orders, while later months show significant increases, indicating strong business expansion.

The consistent rise in revenue suggests that growth is primarily driven by increased order volume rather than fluctuations in pricing.

## Step 6.3 — Orders by Day of Week

### Objective
Analyze how order volume varies across days of the week.

### Approach
Extract day of week from order_purchase_timestamp and count total orders.

Which days generate the highest and lowest order activity?

In [19]:
#Total Orders (again for reference)

orders = con.execute("SELECT * FROM orders LIMIT 5").df();
orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [27]:
con.execute("""SELECT strftime('%A',order_purchase_timestamp) AS Day_Of_Week,
                COUNT(order_id) AS Total_Orders
                FROM orders
                GROUP BY strftime('%A', order_purchase_timestamp)
                ORDER BY strftime('%A', order_purchase_timestamp)""").df()

,Day_Of_Week,Total_Orders
0,Friday,14122
1,Monday,16196
2,Saturday,10887
3,Sunday,11960
4,Thursday,14761
5,Tuesday,15963
6,Wednesday,15552


#### Conclusion

Order volume is highest on Monday, with a noticeable decline toward the weekend.

Weekdays consistently show higher order activity compared to weekends, indicating stronger customer engagement during working days.

Saturday records the lowest number of orders, suggesting reduced purchasing behavior over the weekend period.

## Step 6.4 — Orders by Hour of Day

### Objective
Analyze how order volume varies across different hours of the day.

### Approach
Extract hour from order_purchase_timestamp and count total orders.

At what time of day are customers most active?

In [67]:
con.execute("""SELECT strftime('%H',order_purchase_timestamp) AS Hours_Of_Day,
                        COUNT(order_id) AS Total_Orders
                        FROM Orders
                        GROUP BY Hours_Of_Day
                        ORDER BY Hours_Of_Day""").df()

,Hours_Of_Day,Total_Orders
0,00,2394
1,01,1170
2,02,510
3,03,272
4,04,206
5,05,188
6,06,502
7,07,1231
8,08,2967
9,09,4785


#### Conclusion
Order volume peaks around late morning (approximately 11:00), representing the highest customer activity.

The lowest activity occurs during early morning hours, particularly around 05:00.

Order volume increases steadily after early morning and remains strong through midday and evening, indicating that customers are most active from late morning to evening hours.

## Step 6.5 — Delivery Time Analysis

### Objective
Measure the average time taken to deliver orders.

### Approach
Calculate the average difference between order_purchase_timestamp and order_delivered_customer_date.

How long does it typically take to deliver an order?

In [44]:
con.execute("""SELECT COUNT(Order_id) AS Total_Orders,
                AVG(DATEDIFF('day', order_purchase_timestamp, order_delivered_customer_date)) AS diff_days
                FROM Orders
                 WHERE order_delivered_customer_date IS NOT NULL
""").df()

,Total_Orders,diff_days
0,96476,12.497336


#### Conclusion
The average delivery time is approximately 12.5 days, indicating the typical fulfillment duration.

This reflects a moderate delivery speed, where logistics operations are functional but not highly optimized for fast delivery.

Reducing delivery time could improve customer satisfaction and increase the likelihood of repeat purchases, making it an important operational improvement area.

## Step 6.6 — Delay Analysis

### Objective
Evaluate whether deliveries are late or on time.

### Approach
Compare actual delivery date with estimated delivery date and calculate average delay and late percentage.

Are orders delivered on time or delayed?

In [22]:
#Total Orders (again for reference)

orders = con.execute("SELECT * FROM orders LIMIT 5").df();
orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [55]:
con.execute("""SELECT  COUNT(Order_id) AS Total_Orders,
                    
                    AVG(DATEDIFF('day', order_estimated_delivery_date, order_delivered_customer_date)) AS Average_Delay,
                       
                SUM(CASE
                       WHEN DATEDIFF('day', order_estimated_delivery_date, order_delivered_customer_date) > 0 
                       THEN 1 ELSE 0 END) * 100.0 / COUNT(Order_id) AS Late_Percentage
                       FROM Orders
                       WHERE order_delivered_customer_date IS NOT NULL""").df()
  

,Total_Orders,Average_Delay,Late_Percentage
0,96476,-11.876881,6.773705


#### Conclusion
The average delivery delay is approximately -11 days, indicating that orders are typically delivered earlier than the estimated delivery date.

Only about 6.7% of orders are delivered late, showing strong delivery reliability.

This suggests that estimated delivery timelines are conservative. Optimizing and slightly reducing estimated delivery times could better align customer expectations with actual performance and enhance user experience.